# Tạo feature dataset — Churn 30 ngày (`churn_30d_feature_schema.md`)

Đọc silver từ MinIO, dựng panel `1 dòng = 1 customer_id × 1 snapshot_month`, tính feature + nhãn theo `churn_30d_feature_schema.md`, lưu local để train H2O.

**Không ghi gold.** Compute trên máy (pandas). MinIO chỉ là nguồn.

| Thành phần | Định nghĩa |
|---|---|
| As-of `t` | Ngày cuối tháng (`snapshot_date`) |
| Feature | Chỉ event `<= t` (gồm lag 1m + rolling 3m trên panel) |
| Population | `signup_date <= t` và chưa Closed tại `t` |
| `label_churn_30d` | **C1** xóa TK `(t, t+30d]` **∨** **C2** free tại `t` + bất hoạt `(t, t+30d]` **∨** **C3** paid→free trong `(t, t+30d]` + bất hoạt `(t, t+30d]` |
| `churn_reason` | `{delete, free_inactive, paid_to_free_inactive, none}` — phân tích, không train |
| Activity | Login/usage/order chủ động — **không** tính marketing `sent_at` |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = None
for _p in [Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent]:
    if (_p / "minio_io.py").exists():
        NOTEBOOK_DIR = _p.resolve()
        sys.path.insert(0, str(NOTEBOOK_DIR))
        break

from minio_io import get_client, load_silver_tables

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR and NOTEBOOK_DIR.name == "notebooks" else Path.cwd()
OUTPUT_PATH = PROJECT_ROOT / "data" / "churn_feature_dataset.parquet"

HORIZON_DAYS = 30
LOOKAHEAD_BUFFER_DAYS = HORIZON_DAYS  # chừa đủ gán label_churn_30d
FREE_TIER = "free"
PAID_TIERS = {"plus", "premium", "basic", "standard"}

## 1. Load silver MinIO

In [ ]:
client = get_client()
tables = load_silver_tables(client)

customers = tables["churn_customers"].copy()
orders = tables["churn_orders"].copy()
payments = tables["churn_payments"].copy()
usage = tables["churn_product_usage"].copy()
subscriptions = tables["churn_subscriptions"].copy()
tickets = tables["churn_support_tickets"].copy()
marketing = tables["churn_marketing_interactions"].copy()
del tables

In [ ]:
def to_id(s: pd.Series) -> pd.Series:
    return s.astype("int64")


customers["customer_id"] = to_id(customers["customer_id"])
for df, cols in [
    (orders, ["order_date"]),
    (payments, ["payment_date"]),
    (usage, ["event_date"]),
    (subscriptions, ["start_date", "end_date"]),
    (tickets, ["created_at"]),
    (marketing, ["sent_at"]),
    (customers, ["birth_date", "signup_date", "closed_date", "last_login_at"]),
]:
    if "customer_id" in df.columns:
        df["customer_id"] = to_id(df["customer_id"])
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["opened", "clicked"]:
    marketing[col] = marketing[col].astype("boolean").fillna(False).astype(bool)

bad_signup = customers["signup_date"] < "2020-01-01"
print("drop sentinel signup_date", int(bad_signup.sum()))
customers = customers.loc[~bad_signup].copy()

subscriptions["tier_norm"] = subscriptions["plan_tier"].astype("string").str.strip().str.lower()
subscriptions["auto_renew"] = subscriptions["auto_renew"].astype("boolean")

## 2. Spine panel `customer_id × snapshot_month`

`churn_customer_snapshot` không nằm silver intern. Spine dựng từ customers + tháng có dữ liệu, chừa `LOOKAHEAD_BUFFER_DAYS` (= 30d) để gán `label_churn_30d`.

In [ ]:
max_event = max(
    usage["event_date"].max(),
    orders["order_date"].max(),
)
last_asof = max_event - pd.Timedelta(days=LOOKAHEAD_BUFFER_DAYS)
start = max(customers["signup_date"].min(), pd.Timestamp("2023-01-01"))

try:
    month_ends = pd.date_range(start, last_asof, freq="ME")
except ValueError:
    month_ends = pd.date_range(start, last_asof, freq="M")

print("max_event", max_event)
print("last_asof", last_asof, f"(buffer={LOOKAHEAD_BUFFER_DAYS}d)")
print("n_months", len(month_ends), "from", month_ends.min(), "to", month_ends.max())

spine = customers.merge(pd.DataFrame({"snapshot_date": month_ends}), how="cross")
spine = spine.loc[spine["signup_date"] <= spine["snapshot_date"]].copy()
closed_before = spine["closed_date"].notna() & (spine["closed_date"] <= spine["snapshot_date"])
spine = spine.loc[~closed_before].copy()
spine["snapshot_month"] = spine["snapshot_date"].dt.to_period("M").dt.to_timestamp("M")
spine["snapshot_month_ord"] = spine["snapshot_date"].dt.year * 12 + spine["snapshot_date"].dt.month
spine = spine.merge(
    customers[["customer_id", "account_status"]].rename(columns={"account_status": "account_status_raw"}),
    on="customer_id",
    how="left",
)
print("spine", spine.shape, "customers", spine["customer_id"].nunique())

## 3. Helpers: recency (`merge_asof`) và cửa sổ thời gian

In [ ]:
def last_event_before(spine_df: pd.DataFrame, events: pd.DataFrame, date_col: str, out_col: str) -> pd.DataFrame:
    left = spine_df[["customer_id", "snapshot_date"]].rename(columns={"snapshot_date": "asof"})
    right = events[["customer_id", date_col]].dropna().rename(columns={date_col: "asof"})
    right[out_col] = right["asof"]
    merged = pd.merge_asof(
        left.sort_values("asof"),
        right.sort_values("asof"),
        by="customer_id",
        on="asof",
        direction="backward",
    )
    return merged.rename(columns={"asof": "snapshot_date"})[["customer_id", "snapshot_date", out_col]]


def next_event_after(spine_df: pd.DataFrame, events: pd.DataFrame, date_col: str, out_col: str) -> pd.DataFrame:
    left = spine_df[["customer_id", "snapshot_date"]].rename(columns={"snapshot_date": "asof"})
    right = events[["customer_id", date_col]].dropna().rename(columns={date_col: "asof"})
    right[out_col] = right["asof"]
    merged = pd.merge_asof(
        left.sort_values("asof"),
        right.sort_values("asof"),
        by="customer_id",
        on="asof",
        direction="forward",
        allow_exact_matches=False,
    )
    return merged.rename(columns={"asof": "snapshot_date"})[["customer_id", "snapshot_date", out_col]]


def window_slice(df: pd.DataFrame, date_col: str, t: pd.Timestamp, days: int) -> pd.DataFrame:
    lo = t - pd.Timedelta(days=days)
    return df.loc[(df[date_col] > lo) & (df[date_col] <= t)]

## 4. Nhóm A — Recency

`last_login_at` hiện tại chỉ dùng khi `<= t`; login lịch sử lấy từ `product_usage.event_type == Login`.

In [ ]:
login_events = usage.loc[usage["event_type"] == "Login", ["customer_id", "event_date"]]
completed_orders = orders.loc[orders["status"] == "Completed", ["customer_id", "order_date"]]

recency = spine[["customer_id", "snapshot_date", "last_login_at", "signup_date"]].copy()
recency = recency.merge(last_event_before(spine, usage, "event_date", "last_usage_at"), on=["customer_id", "snapshot_date"], how="left")
recency = recency.merge(last_event_before(spine, completed_orders, "order_date", "last_order_at"), on=["customer_id", "snapshot_date"], how="left")
recency = recency.merge(last_event_before(spine, login_events, "event_date", "last_usage_login_at"), on=["customer_id", "snapshot_date"], how="left")

login_from_profile = recency["last_login_at"].where(recency["last_login_at"] <= recency["snapshot_date"])
recency["last_login_asof"] = pd.concat(
    [recency["last_usage_login_at"], login_from_profile],
    axis=1,
).max(axis=1)

for src, out in [
    ("last_login_asof", "days_since_last_login"),
    ("last_usage_at", "days_since_last_usage_event"),
    ("last_order_at", "days_since_last_order"),
]:
    recency[out] = (recency["snapshot_date"] - recency[src]).dt.days
    tenure = (recency["snapshot_date"] - recency["signup_date"]).dt.days.clip(lower=0)
    recency[out] = recency[out].fillna(tenure)

recency = recency[["customer_id", "snapshot_date", "days_since_last_login", "days_since_last_usage_event", "days_since_last_order"]]
recency.head()

## 5. Feature theo cửa sổ 7/30/60/90 ngày + subscription history

Lặp theo từng `snapshot_date` để tránh nổ RAM. Một vòng tính recency-window, payments, marketing, subscription lifecycle theo schema.

In [ ]:
activity = pd.concat(
    [
        usage[["customer_id", "event_date"]].rename(columns={"event_date": "activity_date"}),
        orders[["customer_id", "order_date"]].rename(columns={"order_date": "activity_date"}),
    ],
    ignore_index=True,
)
activity["activity_day"] = activity["activity_date"].dt.normalize()

tickets = tickets.copy()
tickets["resolved_at"] = tickets["created_at"] + pd.to_timedelta(tickets["resolution_hours"], unit="h")

sub_events = subscriptions.dropna(subset=["start_date"]).copy()

window_parts = []
snapshot_dates = np.sort(spine["snapshot_date"].unique())
print("n_snapshots", len(snapshot_dates))

for i, t in enumerate(snapshot_dates, start=1):
    t = pd.Timestamp(t)

    base = spine.loc[spine["snapshot_date"] == t, ["customer_id", "snapshot_date"]].copy()

    u7 = window_slice(usage, "event_date", t, 7)
    u30 = window_slice(usage, "event_date", t, 30)
    u60 = window_slice(usage, "event_date", t, 60)
    o30 = window_slice(orders, "order_date", t, 30)
    o90 = window_slice(orders, "order_date", t, 90)
    p60 = window_slice(payments, "payment_date", t, 60)
    p90 = window_slice(payments, "payment_date", t, 90)
    tk30 = window_slice(tickets, "created_at", t, 30)
    tk90 = window_slice(tickets, "created_at", t, 90)
    mk30 = window_slice(marketing, "sent_at", t, 30)
    mk60 = window_slice(marketing, "sent_at", t, 60)
    act7 = window_slice(activity, "activity_date", t, 7)
    act14 = window_slice(activity, "activity_date", t, 14)
    act30 = window_slice(activity, "activity_date", t, 30)
    act60 = window_slice(activity, "activity_date", t, 60)
    act90 = window_slice(activity, "activity_date", t, 90)

    o_hist = orders.loc[(orders["order_date"] <= t) & (orders["status"] == "Completed")]
    o90_comp = o90.loc[o90["status"] == "Completed"]
    pay_hist = payments.loc[payments["payment_date"] <= t]
    usage_asof = usage.loc[usage["event_date"] <= t]
    usage_cnt_asof = usage_asof.groupby("customer_id").size().rename("usage_cnt_asof_t")
    mkt_asof = marketing.loc[marketing["sent_at"] <= t]
    mkt_all_asof = mkt_asof.groupby("customer_id").agg(
        mkt_sent_all=("interaction_id", "count"),
        mkt_clicked_all=("clicked", "sum"),
    )

    usage_7 = u7.groupby("customer_id").agg(num_usage_events_7d=("usage_id", "count"))
    usage_30 = u30.groupby("customer_id").agg(
        num_usage_events_30d=("usage_id", "count"),
        avg_session_duration_30d=("session_duration_sec", "mean"),
        total_session_time_30d=("session_duration_sec", "sum"),
        event_type_diversity_30d=("event_type", "nunique"),
    )
    usage_60 = u60.groupby("customer_id").agg(
        num_usage_events_60d=("usage_id", "count"),
        total_usage_60d=("usage_id", "count"),
        avg_usage_duration_60d=("session_duration_sec", "mean"),
    )
    orders_30 = o30.groupby("customer_id").agg(orders_last_30d=("order_id", "count"))
    orders_90 = o90.groupby("customer_id").agg(orders_last_90d=("order_id", "count"))
    aov_90 = o90_comp.groupby("customer_id")["total_amount"].mean().rename("avg_order_value_90d")
    pay_60g = p60.groupby("customer_id").agg(
        payments_60d=("payment_id", "count"),
        payments_failed_60d=("status", lambda s: (s == "Failed").sum()),
    )
    pay_90g = p90.groupby("customer_id").agg(
        payments_90d=("payment_id", "count"),
        payments_success_90d=("status", lambda s: (s == "Success").sum()),
        failed_payments_90d=("status", lambda s: (s == "Failed").sum()),
    )
    tick_30 = tk30.loc[tk30["category"] == "Account"].groupby("customer_id").agg(
        tickets_account_30d=("ticket_id", "count"),
    )
    tick_90 = tk90.groupby("customer_id").agg(
        num_tickets_90d=("ticket_id", "count"),
        avg_csat_score=("csat_score", "mean"),
        tickets_about_cancel_90d=("category", lambda s: (s == "Account").sum()),
    )
    open_tk = tickets.loc[
        (tickets["created_at"] <= t)
        & (tickets["resolved_at"].isna() | (tickets["resolved_at"] > t))
    ]
    open_flag = open_tk.groupby("customer_id").size().rename("open_ticket_n")
    mkt_30 = mk30.groupby("customer_id").agg(
        mkt_sent_30d=("interaction_id", "count"),
        mkt_opened_30d=("opened", "sum"),
        mkt_clicked_30d=("clicked", "sum"),
    )
    mkt_60g = mk60.groupby("customer_id").agg(
        mkt_sent_60d=("interaction_id", "count"),
        mkt_opened_60d=("opened", "sum"),
        mkt_clicked_60d=("clicked", "sum"),
    )
    spend = o_hist.groupby("customer_id")["total_amount"].sum().rename("total_spend_to_date")
    has_completed = o_hist.groupby("customer_id").size().rename("has_completed_order_n")
    ad7 = act7.groupby("customer_id")["activity_day"].nunique().rename("total_active_days_7d")
    ad30 = act30.groupby("customer_id")["activity_day"].nunique().rename("total_active_days_30d")
    ad60 = act60.groupby("customer_id")["activity_day"].nunique().rename("total_active_days_60d")
    ad90 = act90.groupby("customer_id")["activity_day"].nunique().rename("total_active_days_90d")
    has_act7 = act7.groupby("customer_id").size().rename("activity_events_7d")
    has_act14 = act14.groupby("customer_id").size().rename("activity_events_14d")
    has_act30 = act30.groupby("customer_id").size().rename("activity_events_30d")

    # subscription as-of t (§18.3): không còn dòng mở → implicit Free
    sub_left = base[["customer_id", "snapshot_date"]].rename(columns={"snapshot_date": "asof"})
    sub_right = sub_events.copy()
    sub_right["asof"] = sub_right["start_date"]
    sub_asof = pd.merge_asof(
        sub_left.sort_values("asof"),
        sub_right.sort_values("asof"),
        by="customer_id",
        on="asof",
        direction="backward",
    ).rename(columns={"asof": "snapshot_date"})
    still_open = sub_asof["end_date"].isna() | (sub_asof["end_date"] > sub_asof["snapshot_date"])
    has_sub_row = sub_asof["start_date"].notna()
    sub_asof["subscription_expired"] = (~still_open | ~has_sub_row).astype(int)
    sub_asof = sub_asof.rename(columns={"plan_tier": "subscription_tier"})
    sub_asof.loc[sub_asof["subscription_expired"] == 1, "subscription_tier"] = "Free"
    sub_asof.loc[sub_asof["subscription_expired"] == 1, "tier_norm"] = FREE_TIER
    sub_asof.loc[sub_asof["subscription_expired"] == 1, "auto_renew"] = False
    sub_asof["days_on_current_tier"] = (sub_asof["snapshot_date"] - sub_asof["start_date"]).dt.days
    sub_asof.loc[sub_asof["subscription_expired"] == 1, "days_on_current_tier"] = np.nan
    sub_asof["days_until_subscription_end"] = (sub_asof["end_date"] - sub_asof["snapshot_date"]).dt.days
    sub_asof["subscription_age_days"] = sub_asof["days_on_current_tier"]

    lo30, lo90 = t - pd.Timedelta(days=30), t - pd.Timedelta(days=90)
    sub_hist_30 = sub_events.loc[(sub_events["start_date"] > lo30) & (sub_events["start_date"] <= t)]
    sub_hist_90 = sub_events.loc[(sub_events["start_date"] > lo90) & (sub_events["start_date"] <= t)]
    down30 = sub_hist_30.loc[
        (sub_hist_30["change_type"] == "Downgrade") & (sub_hist_30["tier_norm"] == FREE_TIER)
    ].groupby("customer_id").size().rename("had_downgrade_30d_n")
    down90 = sub_hist_90.loc[
        (sub_hist_90["change_type"] == "Downgrade") & (sub_hist_90["tier_norm"] == FREE_TIER)
    ].groupby("customer_id").size().rename("had_downgrade_90d_n")
    up90 = sub_hist_90.loc[sub_hist_90["change_type"] == "Upgrade"].groupby("customer_id").size().rename("had_upgrade_90d_n")
    plan_chg90 = sub_hist_90.groupby("customer_id").size().rename("plan_changes_90d")
    last_down = (
        sub_events.loc[
            (sub_events["change_type"] == "Downgrade")
            & (sub_events["tier_norm"] == FREE_TIER)
            & (sub_events["start_date"] <= t)
        ]
        .sort_values("start_date")
        .groupby("customer_id")["start_date"]
        .last()
        .rename("last_downgrade_date")
    )
    last_pay = (
        pay_hist.sort_values("payment_date")
        .groupby("customer_id")["payment_date"]
        .last()
        .rename("last_payment_date")
    )

    sub_feat = sub_asof[[
        "customer_id", "snapshot_date", "subscription_tier", "auto_renew",
        "days_on_current_tier", "days_until_subscription_end", "subscription_expired",
        "subscription_age_days", "tier_norm",
    ]]
    sub_feat = sub_feat.merge(down30, on="customer_id", how="left")
    sub_feat = sub_feat.merge(down90, on="customer_id", how="left")
    sub_feat = sub_feat.merge(up90, on="customer_id", how="left")
    sub_feat = sub_feat.merge(plan_chg90, on="customer_id", how="left")
    sub_feat = sub_feat.merge(last_down, on="customer_id", how="left")
    sub_feat = sub_feat.merge(last_pay, on="customer_id", how="left")
    sub_feat["days_since_last_downgrade"] = (t - sub_feat["last_downgrade_date"]).dt.days
    sub_feat["days_since_last_payment"] = (t - sub_feat["last_payment_date"]).dt.days

    feat = base.merge(usage_7, on="customer_id", how="left")
    feat = feat.merge(usage_30, on="customer_id", how="left")
    feat = feat.merge(usage_60, on="customer_id", how="left")
    feat = feat.merge(orders_30, on="customer_id", how="left")
    feat = feat.merge(orders_90, on="customer_id", how="left")
    feat = feat.merge(aov_90, on="customer_id", how="left")
    feat = feat.merge(pay_60g, on="customer_id", how="left")
    feat = feat.merge(pay_90g, on="customer_id", how="left")
    feat = feat.merge(tick_30, on="customer_id", how="left")
    feat = feat.merge(tick_90, on="customer_id", how="left")
    feat = feat.merge(open_flag, on="customer_id", how="left")
    feat = feat.merge(mkt_30, on="customer_id", how="left")
    feat = feat.merge(mkt_60g, on="customer_id", how="left")
    feat = feat.merge(spend, on="customer_id", how="left")
    feat = feat.merge(has_completed, on="customer_id", how="left")
    feat = feat.merge(ad7, on="customer_id", how="left")
    feat = feat.merge(ad30, on="customer_id", how="left")
    feat = feat.merge(ad60, on="customer_id", how="left")
    feat = feat.merge(ad90, on="customer_id", how="left")
    feat = feat.merge(has_act7, on="customer_id", how="left")
    feat = feat.merge(has_act14, on="customer_id", how="left")
    feat = feat.merge(has_act30, on="customer_id", how="left")
    feat = feat.merge(sub_feat, on=["customer_id", "snapshot_date"], how="left")
    feat = feat.merge(usage_cnt_asof, on="customer_id", how="left")
    feat = feat.merge(mkt_all_asof, on="customer_id", how="left")

    window_parts.append(feat)
    if i == 1 or i % 6 == 0 or i == len(snapshot_dates):
        print(f"  snapshot {i}/{len(snapshot_dates)} {t.date()} rows={len(feat):,}")

window_df = pd.concat(window_parts, ignore_index=True)
print("window_df", window_df.shape)

In [ ]:
count_zero_cols = [
    "num_usage_events_7d", "num_usage_events_30d", "num_usage_events_60d",
    "total_usage_60d", "event_type_diversity_30d",
    "orders_last_30d", "orders_last_90d",
    "payments_60d", "payments_failed_60d", "payments_90d", "payments_success_90d", "failed_payments_90d",
    "num_tickets_90d", "tickets_account_30d", "tickets_about_cancel_90d", "open_ticket_n",
    "mkt_sent_30d", "mkt_opened_30d", "mkt_clicked_30d",
    "mkt_sent_60d", "mkt_opened_60d", "mkt_clicked_60d",
    "mkt_sent_all", "mkt_opened_all", "mkt_clicked_all",
    "total_spend_to_date", "has_completed_order_n", "usage_cnt_asof_t",
    "total_active_days_7d", "total_active_days_30d", "total_active_days_60d", "total_active_days_90d",
    "activity_events_7d", "activity_events_14d", "activity_events_30d",
    "had_downgrade_30d_n", "had_downgrade_90d_n", "had_upgrade_90d_n", "plan_changes_90d",
    "total_session_time_30d", "mkt_sent_all", "mkt_clicked_all",
]
for col in count_zero_cols:
    if col in window_df.columns:
        window_df[col] = window_df[col].fillna(0)

window_df["has_any_activity_7d"] = (window_df["activity_events_7d"] > 0).astype(int)
window_df["has_any_activity_14d"] = (window_df["activity_events_14d"] > 0).astype(int)
window_df["has_any_activity_30d"] = (window_df["activity_events_30d"] > 0).astype(int)
window_df["has_unresolved_ticket"] = (window_df["open_ticket_n"] > 0).astype(int)
window_df["has_marketing_click_30d"] = (window_df["mkt_clicked_30d"] > 0).astype(int)
window_df["has_completed_order"] = (window_df["has_completed_order_n"] > 0).astype(int)
window_df["had_downgrade_30d"] = (window_df["had_downgrade_30d_n"] > 0).astype(int)
window_df["had_downgrade_90d"] = (window_df["had_downgrade_90d_n"] > 0).astype(int)
window_df["had_upgrade_90d"] = (window_df["had_upgrade_90d_n"] > 0).astype(int)

window_df["subscription_tier"] = window_df["subscription_tier"].fillna("Free")
window_df["tier_norm"] = window_df["tier_norm"].fillna(FREE_TIER)
window_df["is_free_tier"] = (window_df["tier_norm"] == FREE_TIER).astype(int)
window_df["is_paid_tier"] = (
    (window_df["subscription_expired"] == 0) & (window_df["tier_norm"] != FREE_TIER)
).astype(int)
window_df["auto_renew"] = window_df["auto_renew"].astype("boolean").fillna(False).astype(int)

window_df["payments_success_rate"] = np.where(
    window_df["payments_90d"] > 0,
    window_df["payments_success_90d"] / window_df["payments_90d"],
    np.nan,
)
window_df["payments_success_rate_missing"] = window_df["payments_success_rate"].isna().astype(int)
window_df["failed_payment_rate_60d"] = np.where(
    window_df["payments_60d"] > 0,
    window_df["payments_failed_60d"] / window_df["payments_60d"],
    np.nan,
)
window_df["avg_csat_score_missing"] = window_df["avg_csat_score"].isna().astype(int)

window_df["open_rate_30d"] = np.where(
    window_df["mkt_sent_30d"] > 0,
    window_df["mkt_opened_30d"] / window_df["mkt_sent_30d"],
    np.nan,
)
window_df["opened_rate_60d"] = np.where(
    window_df["mkt_sent_60d"] > 0,
    window_df["mkt_opened_60d"] / window_df["mkt_sent_60d"],
    np.nan,
)
window_df["clicked_rate_all_time"] = np.where(
    window_df["mkt_sent_all"] > 0,
    window_df["mkt_clicked_all"] / window_df["mkt_sent_all"],
    np.nan,
)
window_df["usage_7d_over_30d"] = window_df["num_usage_events_7d"] / (window_df["num_usage_events_30d"] + 1)
window_df["usage_60d_share"] = window_df["num_usage_events_60d"] / (window_df["usage_cnt_asof_t"] + 1)

# Proxy cancel intent (không có event cancel riêng)
window_df["cancel_request_flag_30d"] = (window_df["tickets_account_30d"] > 0).astype(int)
window_df["had_cancel_attempt_90d"] = (
    (window_df["tickets_about_cancel_90d"] > 0) | (window_df["had_downgrade_90d"] == 1)
).astype(int)

drop_tmp = [
    "tier_norm", "activity_events_7d", "activity_events_14d", "activity_events_30d",
    "had_downgrade_30d_n", "had_downgrade_90d_n", "had_upgrade_90d_n",
    "open_ticket_n", "has_completed_order_n", "usage_cnt_asof_t",
    "mkt_sent_30d", "mkt_opened_30d", "mkt_clicked_30d",
    "mkt_sent_60d", "mkt_opened_60d", "mkt_clicked_60d",
    "mkt_sent_all", "mkt_clicked_all",
    "payments_60d", "payments_failed_60d", "payments_90d", "payments_success_90d",
    "tickets_account_30d", "last_downgrade_date", "last_payment_date",
]
window_df = window_df.drop(columns=[c for c in drop_tmp if c in window_df.columns])

## 6. Momentum theo tháng lịch (panel)

Lag 1m + rolling 3m tính sau khi join panel. Thêm đổi open/click rate và usage duration theo tháng.

In [ ]:
usage["event_month"] = usage["event_date"].dt.to_period("M")
monthly_usage = (
    usage.groupby(["customer_id", "event_month"], as_index=False)
    .agg(
        usage_month_cnt=("usage_id", "count"),
        session_month_avg=("session_duration_sec", "mean"),
    )
)

marketing["event_month"] = marketing["sent_at"].dt.to_period("M")
monthly_mkt = (
    marketing.groupby(["customer_id", "event_month"], as_index=False)
    .agg(mkt_sent=("interaction_id", "count"), mkt_opened=("opened", "sum"), mkt_clicked=("clicked", "sum"))
)
monthly_mkt["open_rate_m"] = monthly_mkt["mkt_opened"] / (monthly_mkt["mkt_sent"] + 1)
monthly_mkt["click_rate_m"] = monthly_mkt["mkt_clicked"] / (monthly_mkt["mkt_sent"] + 1)

trend = spine[["customer_id", "snapshot_date"]].copy()
trend["event_month"] = pd.to_datetime(trend["snapshot_date"]).dt.to_period("M")
trend = trend.merge(monthly_usage, on=["customer_id", "event_month"], how="left")
trend = trend.merge(monthly_mkt, on=["customer_id", "event_month"], how="left")

prev = monthly_usage.rename(columns={
    "event_month": "prev_month",
    "usage_month_cnt": "usage_prev_cnt",
    "session_month_avg": "session_prev_avg",
})
trend["prev_month"] = trend["event_month"] - 1
trend = trend.merge(prev, on=["customer_id", "prev_month"], how="left")

prev2 = monthly_usage.rename(columns={
    "event_month": "prev2_month",
    "usage_month_cnt": "usage_prev2_cnt",
})
trend["prev2_month"] = trend["event_month"] - 2
trend = trend.merge(prev2[["customer_id", "prev2_month", "usage_prev2_cnt"]], on=["customer_id", "prev2_month"], how="left")

prev_mkt = monthly_mkt.rename(columns={
    "event_month": "prev_month",
    "open_rate_m": "open_rate_prev",
    "click_rate_m": "click_rate_prev",
})
trend = trend.merge(prev_mkt[["customer_id", "prev_month", "open_rate_prev", "click_rate_prev"]], on=["customer_id", "prev_month"], how="left")

for col in ["usage_month_cnt", "usage_prev_cnt", "usage_prev2_cnt"]:
    trend[col] = trend[col].fillna(0)

trend["usage_trend_30d"] = trend["usage_month_cnt"] / (trend["usage_prev_cnt"] + 1) - 1
trend["session_duration_trend"] = trend["session_month_avg"] - trend["session_prev_avg"]
trend["session_duration_trend_missing"] = trend["session_duration_trend"].isna().astype(int)
trend["activity_slope_3m"] = (trend["usage_month_cnt"] - trend["usage_prev2_cnt"]) / 2.0
trend["is_declining_engagement"] = (
    (trend["usage_month_cnt"] < trend["usage_prev_cnt"])
    & (trend["usage_prev_cnt"] < trend["usage_prev2_cnt"])
).astype(int)
trend["opened_rate_change"] = trend["open_rate_m"] - trend["open_rate_prev"]
trend["clicked_rate_change"] = trend["click_rate_m"] - trend["click_rate_prev"]
trend["usage_duration_change"] = trend["session_month_avg"] - trend["session_prev_avg"]

trend = trend[[
    "customer_id", "snapshot_date",
    "usage_trend_30d", "session_duration_trend", "session_duration_trend_missing",
    "activity_slope_3m", "is_declining_engagement",
    "opened_rate_change", "clicked_rate_change", "usage_duration_change",
]]
trend.head()

## 7. Demographics & lifecycle (static / slow-changing)

In [ ]:
profile = spine[[
    "customer_id", "snapshot_date", "snapshot_month", "snapshot_month_ord",
    "gender", "region", "city", "birth_date", "signup_date", "account_status_raw", "closed_date",
]].copy()
profile["tenure_days"] = (profile["snapshot_date"] - profile["signup_date"]).dt.days.clip(lower=0)
profile["age"] = ((profile["snapshot_date"] - profile["birth_date"]).dt.days / 365.25).round(1)

# Reconstruct account_status as-of t (§18.4) — không copy Closed tương lai
raw_status = profile["account_status_raw"].fillna("Active")
future_closed = raw_status.eq("Closed") & (
    profile["closed_date"].isna() | (profile["closed_date"] > profile["snapshot_date"])
)
profile["account_status_at_t"] = raw_status.where(~future_closed, "Active")
profile["account_status_at_t"] = profile["account_status_at_t"].where(
    profile["account_status_at_t"] != "Closed", "Active"
)
profile = profile.drop(columns=["account_status_raw", "closed_date"])
profile.head()

## 8. Nhãn `label_churn_30d` + `churn_reason` (C1 / C2 / C3)

- **C1:** `closed_date ∈ (t, t+30d]`
- **C2:** `is_free_tier` tại `t` và không activity trong `(t, t+30d]`
- **C3:** `is_paid_tier` tại `t`, downgrade paid→free trong `(t, t+30d]`, và không activity trong `(t, t+30d]`

`churn_reason` ưu tiên: `delete` > `paid_to_free_inactive` > `free_inactive` > `none`

In [ ]:
activity_future = pd.concat(
    [
        usage[["customer_id", "event_date"]].rename(columns={"event_date": "activity_date"}),
        orders[["customer_id", "order_date"]].rename(columns={"order_date": "activity_date"}),
    ],
    ignore_index=True,
).dropna()

# Activity đầu tiên sau t (để kiểm tra bất hoạt trong horizon 30d)
asof_left = spine[["customer_id", "snapshot_date"]].rename(columns={"snapshot_date": "asof"})
act_after_t = activity_future.rename(columns={"activity_date": "asof"}).copy()
act_after_t["next_activity_after_t"] = act_after_t["asof"]
next_act_after_t = pd.merge_asof(
    asof_left.sort_values("asof"),
    act_after_t.sort_values("asof"),
    by="customer_id",
    on="asof",
    direction="forward",
    allow_exact_matches=False,
).rename(columns={"asof": "snapshot_date"})[["customer_id", "snapshot_date", "next_activity_after_t"]]

# Downgrade paid→free trong horizon
downgrade_events = sub_events.loc[
    (sub_events["change_type"] == "Downgrade") & (sub_events["tier_norm"] == FREE_TIER),
    ["customer_id", "start_date"],
].rename(columns={"start_date": "asof"})
downgrade_events["downgrade_date"] = downgrade_events["asof"]
downgrade_events = downgrade_events.sort_values(["customer_id", "asof"]).drop_duplicates(["customer_id", "asof"])

next_downgrade = pd.merge_asof(
    asof_left.sort_values("asof"),
    downgrade_events.sort_values("asof"),
    by="customer_id",
    on="asof",
    direction="forward",
    allow_exact_matches=False,
).rename(columns={"asof": "snapshot_date"})

label_base = spine[["customer_id", "snapshot_date", "closed_date"]].merge(
    window_df[["customer_id", "snapshot_date", "is_free_tier", "is_paid_tier"]],
    on=["customer_id", "snapshot_date"],
    how="left",
)
labels = label_base.merge(next_act_after_t, on=["customer_id", "snapshot_date"], how="left")
labels = labels.merge(next_downgrade[["customer_id", "snapshot_date", "downgrade_date"]], on=["customer_id", "snapshot_date"], how="left")

t_end = labels["snapshot_date"] + pd.Timedelta(days=HORIZON_DAYS)
inactive_horizon = labels["next_activity_after_t"].isna() | (labels["next_activity_after_t"] > t_end)

c1 = labels["closed_date"].notna() & (labels["closed_date"] > labels["snapshot_date"]) & (labels["closed_date"] <= t_end)
c2 = (labels["is_free_tier"] == 1) & inactive_horizon
downgrade_in_h = (
    (labels["is_paid_tier"] == 1)
    & labels["downgrade_date"].notna()
    & (labels["downgrade_date"] > labels["snapshot_date"])
    & (labels["downgrade_date"] <= t_end)
)
c3 = downgrade_in_h & inactive_horizon

labels["label_churn_30d"] = (c1 | c2 | c3).astype(int)
labels["churn_reason"] = "none"
labels.loc[c2 & ~c1 & ~c3, "churn_reason"] = "free_inactive"
labels.loc[c3 & ~c1, "churn_reason"] = "paid_to_free_inactive"
labels.loc[c1, "churn_reason"] = "delete"

labels = labels[["customer_id", "snapshot_date", "label_churn_30d", "churn_reason"]]
print(labels["label_churn_30d"].mean(), "churn rate")
print(labels["churn_reason"].value_counts())
labels.head()


## 9. Join dataset + lag/rolling + derived + lưu local

Xuất đúng cột theo `churn_30d_feature_schema.md` (keys, label, meta, features P0–P2, derived).

In [ ]:
dataset = profile.merge(recency, on=["customer_id", "snapshot_date"], how="left")
dataset = dataset.merge(window_df, on=["customer_id", "snapshot_date"], how="left")
dataset = dataset.merge(trend, on=["customer_id", "snapshot_date"], how="left")
dataset = dataset.merge(labels, on=["customer_id", "snapshot_date"], how="left")

tenure_months = (dataset["tenure_days"] / 30.0).clip(lower=1)
dataset["avg_spend_to_date_per_month"] = dataset["total_spend_to_date"].fillna(0) / tenure_months

dataset = dataset.sort_values(["customer_id", "snapshot_date"]).reset_index(drop=True)
dataset["days_since_last_activity"] = dataset[
    ["days_since_last_login", "days_since_last_usage_event", "days_since_last_order"]
].min(axis=1)
dataset["days_since_last_completed_order"] = dataset["days_since_last_order"]
dataset["activity_gap_ratio"] = dataset["days_since_last_activity"] / (dataset["tenure_days"] + 1)
dataset["days_since_last_downgrade"] = dataset["days_since_last_downgrade"].fillna(dataset["tenure_days"])
dataset["days_since_last_payment"] = dataset["days_since_last_payment"].fillna(dataset["tenure_days"])

g = dataset.groupby("customer_id", sort=False)
dataset["num_usage_events_30d_lag1m"] = g["num_usage_events_30d"].shift(1)
dataset["days_since_last_activity_lag1m"] = g["days_since_last_activity"].shift(1)
dataset["days_since_last_activity_diff1"] = dataset["days_since_last_activity"] - dataset["days_since_last_activity_lag1m"]
dataset["num_usage_events_roll3m_sum"] = g["num_usage_events_30d"].transform(lambda s: s.rolling(3, min_periods=1).sum())
dataset["avg_session_duration_roll3m_mean"] = g["avg_session_duration_30d"].transform(lambda s: s.rolling(3, min_periods=1).mean())
dataset["orders_roll3m_sum"] = g["orders_last_30d"].transform(lambda s: s.rolling(3, min_periods=1).sum())

dataset["reactivation_flag"] = (
    dataset["days_since_last_activity_lag1m"].gt(30)
    & dataset["days_since_last_activity"].le(30)
).fillna(False).astype(int)

# Derived interactions (schema §12)
dataset["free_and_inactive_14d"] = ((dataset["is_free_tier"] == 1) & (dataset["days_since_last_activity"] >= 14)).astype(int)
dataset["free_and_inactive_21d"] = ((dataset["is_free_tier"] == 1) & (dataset["days_since_last_activity"] >= 21)).astype(int)
dataset["paid_weak_engagement"] = ((dataset["is_paid_tier"] == 1) & (dataset["days_since_last_activity"] >= 14)).astype(int)
dataset["recent_downgrade_and_quiet"] = ((dataset["had_downgrade_30d"] == 1) & (dataset["days_since_last_activity"] >= 7)).astype(int)
dataset["auto_renew_off_paid"] = ((dataset["is_paid_tier"] == 1) & (dataset["auto_renew"] == 0)).astype(int)

bins = [-np.inf, 7, 14, 21, 29, np.inf]
labels_bucket = ["0-7", "8-14", "15-21", "22-29", "30+"]
dataset["risk_recency_bucket"] = pd.cut(dataset["days_since_last_activity"], bins=bins, labels=labels_bucket).astype("string")

dataset["customer_id"] = dataset["customer_id"].astype(str)
dataset = dataset.drop(columns=["total_spend_to_date"], errors="ignore")

SCHEMA_COLUMNS = [
    "customer_id", "snapshot_date", "snapshot_month", "snapshot_month_ord",
    "label_churn_30d", "churn_reason",
    # P0 recency / activity
    "days_since_last_activity", "days_since_last_login", "days_since_last_usage_event", "days_since_last_order",
    "has_any_activity_7d", "has_any_activity_14d", "has_any_activity_30d",
    "total_active_days_7d", "total_active_days_30d", "total_active_days_60d", "total_active_days_90d",
    "activity_gap_ratio",
    # P0 subscription
    "subscription_tier", "is_free_tier", "is_paid_tier", "auto_renew",
    "days_on_current_tier", "days_until_subscription_end", "subscription_expired", "subscription_age_days",
    "had_downgrade_30d", "had_downgrade_90d", "had_upgrade_90d", "plan_changes_90d", "days_since_last_downgrade",
    # P0 account intent
    "cancel_request_flag_30d", "had_cancel_attempt_90d", "account_status_at_t",
    # P1 usage
    "num_usage_events_7d", "num_usage_events_30d", "num_usage_events_60d",
    "usage_7d_over_30d", "usage_60d_share",
    "avg_session_duration_30d", "total_session_time_30d", "event_type_diversity_30d",
    "total_usage_60d", "avg_usage_duration_60d",
    # P1 momentum
    "usage_trend_30d", "session_duration_trend", "session_duration_trend_missing",
    "activity_slope_3m", "is_declining_engagement", "reactivation_flag",
    "opened_rate_change", "clicked_rate_change", "usage_duration_change",
    "days_since_last_activity_lag1m", "days_since_last_activity_diff1",
    "num_usage_events_30d_lag1m", "num_usage_events_roll3m_sum",
    "avg_session_duration_roll3m_mean", "orders_roll3m_sum",
    # P1 orders / payments
    "orders_last_30d", "orders_last_90d", "avg_order_value_90d",
    "has_completed_order", "days_since_last_completed_order", "days_since_last_payment",
    "payments_success_rate", "payments_success_rate_missing",
    "failed_payment_rate_60d", "failed_payments_90d", "avg_spend_to_date_per_month",
    # P1 support
    "num_tickets_90d", "has_unresolved_ticket", "avg_csat_score", "avg_csat_score_missing",
    "tickets_about_cancel_90d",
    # P2 lifecycle / demo
    "tenure_days", "age", "gender", "region", "city",
    # P2 marketing
    "open_rate_30d", "has_marketing_click_30d", "opened_rate_60d", "clicked_rate_all_time",
    # Derived
    "free_and_inactive_14d", "free_and_inactive_21d", "paid_weak_engagement",
    "recent_downgrade_and_quiet", "auto_renew_off_paid", "risk_recency_bucket",
]

missing_cols = [c for c in SCHEMA_COLUMNS if c not in dataset.columns]
if missing_cols:
    raise KeyError(f"Missing schema columns: {missing_cols}")

dataset = dataset[SCHEMA_COLUMNS].sort_values(["customer_id", "snapshot_date"]).reset_index(drop=True)
print(dataset.shape)
print("unique pairs", dataset.groupby(["customer_id", "snapshot_month"]).ngroups)
dataset.head()

In [ ]:
assert dataset.duplicated(["customer_id", "snapshot_month"]).sum() == 0
assert dataset["label_churn_30d"].isin([0, 1]).all()
assert set(dataset["churn_reason"].unique()) <= {"delete", "free_inactive", "paid_to_free_inactive", "none"}
assert (dataset["label_churn_30d"] == 1).eq(dataset["churn_reason"] != "none").all()
assert (dataset["is_free_tier"] + dataset["is_paid_tier"] <= 1).all()
assert dataset["account_status_at_t"].ne("Closed").all()
assert dataset["is_free_tier"].isin([0, 1]).all()
assert dataset["is_paid_tier"].isin([0, 1]).all()
assert (dataset["tenure_days"] >= 0).all()
print("QA ok — columns", len(dataset.columns))
print("label_churn_30d rate", dataset["label_churn_30d"].mean().round(4))
print(dataset["churn_reason"].value_counts())
print("\nnull % top 15")
print((dataset.isna().mean() * 100).sort_values(ascending=False).head(15).round(2).to_string())

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
dataset.to_parquet(OUTPUT_PATH, index=False)
print("saved", OUTPUT_PATH.resolve(), "rows", len(dataset))

In [ ]:
OUTPUT_CSV = OUTPUT_PATH.with_suffix(".csv")
dataset.to_csv(OUTPUT_CSV, index=False)
print("saved", OUTPUT_CSV.resolve(), "rows", len(dataset))